# Dissertation §3.5.3 — reproducible workflow

This notebook is a thin wrapper around the installable package
[`caspian_fish_quality`](../README.md). All logic lives under `src/`;
run the reproduction script to regenerate tables for the dissertation.

**Requirements:** Python 3.10–3.12, `pip install -e ".[dev,test]"` from repo root.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "caspian_fish_quality").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import caspian_fish_quality as cfq
from caspian_fish_quality.config import get_settings

print(f"Package version: {cfq.__version__}")
print(f"Settings: seed={get_settings().seed}, n_per_group={get_settings().n_per_group}")

## 2. Reproduce section 3.5.3 (full pipeline)

Runs synthetic generation (N=2000), ML grid, and sturgeon transfer;
writes CSV + `tables_az.md` under `results/section_3_5_3/` (~1–2 min).

In [ ]:
%run ../scripts/reproduce_section_3_5_3.py

## 3. Inspect outputs

In [ ]:
from IPython.display import Markdown, display
import pandas as pd

OUT = ROOT / "results" / "section_3_5_3"

display(Markdown((OUT / "tables_az.md").read_text(encoding="utf-8")))

print("\n--- Table 3.5.6 (CV R²) ---")
display(pd.read_csv(OUT / "regression_cv_r2.csv"))

print("\n--- Table 3.5.7 (transfer) ---")
display(pd.read_csv(OUT / "transfer_table_3_5_7.csv"))

## 4. Optional: quick API smoke test

In [ ]:
from caspian_fish_quality.synth.generators import load_default_df_dict, generate_all_synthetic
from caspian_fish_quality.merge import merge_static, merge_storage
from caspian_fish_quality.ml.datasets import generate_phd_dataset, prepare_full_dataset, run_all
from caspian_fish_quality.transfer import run_transfer_test

seed = get_settings().seed
n = get_settings().n_per_group

syn = generate_all_synthetic(load_default_df_dict(), n_per_group=n, seed=seed, verbose=False)
static_df = merge_static(syn)
storage_df = merge_storage(syn)
phd_df = generate_phd_dataset(n_per_group=n, seed=seed)
full = prepare_full_dataset(phd_df, static_df, storage_df)
results, _ = run_all(full, random_state=seed)
transfer = run_transfer_test(phd_df, static_df, storage_df)

print(f"ML rows: {len(results)}, transfer rows: {len(transfer)}")